# PaliGemma-3B Image Phrase Extractor — Google Colab (Google Drive)

**Model:** PaliGemma 3B Mix 224 (google/paligemma-3b-mix-224) — publicly available, HuggingFace login required.

Link:https://huggingface.co/google/paligemma-3b-mix-224

• 3B Parameters

• 2024-2025 years model

+lightweight

-Limited reasoning ability, struggles with complex prompts.




**Output per image (saved to phrases_paligemma.csv):**

**filename	phrase	meaningful_tokens	word_count	meaningful_token_count**

---


*   **word_count** — total words in the phrase
*   **meaningful_token_count** — words after removing stopwords (a, the, is, in, of, ...)


### **Before running**:
* Upload your image folder to Google Drive → My Drive
* Set image_folder path correctly in the notebook
* Allow popups for colab.research.google.com

**If Drive mount fails:**
* Enable popups and retry
* Or: Runtime → Restart session

**Runtime recommendation:** Runtime → Change runtime type → T4 GPU

In [8]:
# ── Cell 0: Log In with your huggingface token ───────────────────────────
from huggingface_hub import login
login()

In [9]:
# ── Cell 1: Mount Google Drive and set image folder ───────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# ↓ Change this to the name of the folder you uploaded to My Drive
DRIVE_IMAGE_FOLDER = 'fire_detection/PickTheModelDataSet(30pics)'
#image_folder = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)"

IMAGE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(IMAGE_DIR), (
    f'Folder not found: {IMAGE_DIR}\n'
    f'Make sure it exists in My Drive.'
)

image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".png", ".jpeg", "webp")) and not f.startswith("._")
]

print(f'Found {len(image_files)} images')

Mounted at /content/drive
Found 30 images


In [10]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate pillow sentencepiece spacy
!python -m spacy download en_core_web_sm

# ⚠️ After this cell finishes, go to Runtime → Restart session,
# then run all cells again from Cell 1.
print('Done. Now go to Runtime → Restart session, then re-run all cells.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 119.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Done. Now go to Runtime → Restart session, then re-run all cells.


In [11]:
# ── Cell 3: Load paligemma-3b-mix-224 model ────────────────────────────────────────
import torch
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor

model_id = "google/paligemma-3b-mix-224"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU:", torch.cuda.is_available())

model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

print("Model loaded!")

GPU: True


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.26M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

Model loaded!


In [12]:
# ── Cell 4: Load spaCy and helper functions ────────────────────────────────────────
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_meaningful_tokens(text):
    doc = nlp(text)
    return [t.text for t in doc if t.pos_ in ["NOUN", "VERB", "ADJ"]]



In [13]:
# ── Cell 5: Inference function ─────────────────────────────────────────────────────
from PIL import Image

def analyze_image(image_path, prompt):

    image = Image.open(image_path).convert("RGB")

    prompt = "<image>\n" + prompt

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to(DEVICE)

    input_len = inputs["input_ids"].shape[1]

    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7
    )

    generated_tokens = output[0][input_len:]

    result = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    tokens = processor.tokenizer.convert_ids_to_tokens(generated_tokens)

    return result, len(tokens), tokens

In [20]:
# ── Cell 6: Configuration ─────────────────────────────────────────────────────

import os

prompt = "Describe this image briefly"

# IMAGE_TYPE = "Flame / Factory Warehouse Workshop"

In [21]:
# ── Cell 7: Process all images and save CSV ───────────────────────────────
import csv

output_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)/phrases_paligemma.csv"

COLUMNS = [
    "filename",
    "phrase",
    "meaningful_tokens",
    "word_count",
    "meaningful_token_count"
]

with open(output_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNS)
    writer.writeheader()

    print("\n===== Processing images =====")

    for image_file in image_files:
        image_path = os.path.join(IMAGE_DIR, image_file)

        try:
            result, out_tok_num, tokens = analyze_image(image_path, prompt)

            meaningful_tokens = extract_meaningful_tokens(result)
            words = result.split()

            writer.writerow({
                "filename": image_file,
                "phrase": result,
                "meaningful_tokens": " ".join(meaningful_tokens),
                "word_count": len(words),
                "meaningful_token_count": len(meaningful_tokens)
            })

            print(
                f"{image_file} | {result} | "
                f"{' '.join(meaningful_tokens)} | "
                f"{len(words)} | {len(meaningful_tokens)}"
            )

        except Exception as e:
            print(f"{image_file} | ERROR: {e}")

print(f"\nDone. CSV saved to {output_path}")


===== Processing images =====
0087_FL_FWW_00150.jpg | A person taking a picture of a large fire. | person taking picture large fire | 9 | 5
0087_FL_FWW_00100.jpg | A pile of bags that are on fire. | pile bags fire | 8 | 3
0088_FL_FWW_00001.jpg | a fire is burning near a trash bin with the number 200 on it. | fire burning trash bin number | 14 | 5
0089_FL_FWW_00001.jpg | a large metal structure is on fire with a pile of rubble on fire nearby. | large metal structure fire pile rubble fire | 15 | 7
0121_FL_FWW_00126.jpg | A group of firemen are working to put out a fire. | group firemen working put fire | 11 | 5
0132_FL_FWW_00089.jpg | a building that is on fire and smoke coming out of it | building fire smoke coming | 12 | 4
0150_FL_FWW_00256.jpg | a sign on a building reads: 09-09-2023 sat. 04:51. | sign building reads sat | 9 | 4
0229_FL_FWW_00269.jpg | a photo of a house with a lot of smoke | photo house lot smoke | 10 | 4
0328_FL_FWW_00293.jpg | A building with a fire burning at nig